# 2. Consultas SQL e pipeline de dados (medalhão)

**Requisito da vaga:** *Desenvolver consultas SQL e pipelines de dados*.

Percorremos o modelo medalhão do projeto:

1. **Bronze** — `raw_telemetry` (payload original, JSONB) e `raw_events`.
2. **Silver** — `stg_telemetry` (validada/deduplicada) e `int_telemetry` (campos derivados: `thermal_stress_index`).
3. **Gold** — star schema: `dim_time`, `dim_transformer`, `dim_location`, `dim_sensor` + `fact_transformer_measurement`.

Pipeline ELT orquestrado por **dbt** (`dbt/models/silver`, `dbt/models/gold`).

> **Pré-requisito:** banco local com dados (`make db && make migrate && make smoke`)
> e o ML service rodando (`make ml-run &`) — ou apenas `make demo`.
>
> Carregar o helper compartilhado (bootstrap de imports, leitores de dados,
> URLs dos serviços) da primeira célula. `notebooks/common.py`.


In [1]:
import sys
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores')
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/notebooks')
import common
import pandas as pd
pd.set_option('display.max_columns', None)

## Bronze → tabelas operacionais

A ingestão (Go) grava o payload original em `raw_telemetry` e a medição normalizada em `measurements` (deduplicada por `{transformer_id}@{timestamp}`).

In [2]:
rows = common.pg_df('''
SELECT transformer_id, count(*) AS n,
       min(ts) AS first_ts, max(ts) AS last_ts
FROM measurements GROUP BY transformer_id ORDER BY n DESC LIMIT 5
''')
rows

,transformer_id,n,first_ts,last_ts
0,TR-001,6,2026-08-12 06:00:00+00:00,2026-08-12 07:18:30+00:00
1,TR-002,5,2026-08-12 07:18:26+00:00,2026-08-12 07:18:30+00:00
2,TR-003,5,2026-08-12 07:18:26+00:00,2026-08-12 07:18:30+00:00


## Silver — validação e campos derivados

`int_telemetry` adiciona `thermal_stress_index` e margens de temperatura, e recomputa o estado do transformador independentemente do emissor.

In [3]:
common.pg_df('''
SELECT transformer_id, state_recomputed, count(*) AS n,
       round(avg(thermal_stress_index)::numeric, 4) AS avg_tsi
FROM int_telemetry GROUP BY 1,2 ORDER BY n DESC LIMIT 5
''')

,transformer_id,state_recomputed,n,avg_tsi
0,TR-001,NORMAL,6,0.2013
1,TR-002,NORMAL,5,0.2456
2,TR-003,NORMAL,5,0.2462


## Gold — star schema analítico

Join entre fato e dimensões para análise por transformador/aplicação.

In [4]:
common.pg_df('''
SELECT dt.transformer_key, dt.application, count(*) AS n_measurements,
       round(avg(fm.oil_temperature_c)::numeric, 2) AS avg_oil_temp_c
FROM fact_transformer_measurement fm
JOIN dim_transformer dt   ON fm.transformer_key   = dt.transformer_key
GROUP BY 1,2 ORDER BY n_measurements DESC LIMIT 6
''')

,transformer_key,application,n_measurements,avg_oil_temp_c
0,TR-001,generation,17,21.71
1,TR-002,industrial,16,21.71
2,TR-003,distribution,13,21.71
3,TR-006,transmission,9,21.70
4,TR-007,transmission,9,21.70
5,TR-005,renewable,9,21.70


## Dril-down de negócio

Consulta analítica: perfil térmico médio por aplicação (região/segmento), reaproveitando `dim_location`.

In [5]:
common.pg_df('''
SELECT dl.region, dt.application,
       round(avg(fm.oil_temperature_c)::numeric, 2) AS avg_oil_temp_c,
       count(*) AS n
FROM fact_transformer_measurement fm
JOIN dim_transformer dt ON fm.transformer_key = dt.transformer_key
JOIN dim_location dl    ON fm.location_key    = dl.location_key
GROUP BY 1,2 ORDER BY 1,2
''')

,region,application,avg_oil_temp_c,n
0,Nordeste,renewable,21.7,9
1,Sudeste,distribution,21.7,22
2,Sudeste,industrial,21.7,25
3,Sudeste,transmission,21.7,18
4,Sul,generation,21.7,35


## Conclusão

- Modelo medalhão bronze→silver→gold com dbt reproduzível (`make dbt`), testes de dados (20 em silver, 25 em gold).
- Consultas analíticas diretas em SQL sobre o star schema — pronto para dashboards.